<a href="https://colab.research.google.com/github/evinracher/3008410-intelligent-systems/blob/main/week7/exercise1/Explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### LLM Explainability

We will use a technique called the "Logit Lens." This allows us to "peek" inside the model’s hidden layers to see how a thought evolves from a random guess in the early layers to a confident answer in the final layer.

In a standard LLM, we only see the final output. But the model has many layers (e.g., 12 layers in GPT-2). With the Logit Lens, we force the model to make a prediction at Layer 1, Layer 6, and Layer 12 to see its "internal brainstorm."

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# 1. Load model and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# 2. Use a very direct prompt
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

# 3. Get the internal states
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    hidden_states = outputs.hidden_states # List of layers
    final_layer_norm = model.transformer.ln_f # This is the "Filter" we need!

def get_interpretable_guesses(layer_index, k=2):
    # Get the raw hidden state
    h = hidden_states[layer_index][0, -1, :]

    # CRITICAL STEP: Apply the model's final normalization to the hidden state
    # This "cleans" the data so the head can read it properly
    h_cleaned = final_layer_norm(h)

    # Project to vocabulary
    logits = model.lm_head(h_cleaned)
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, k)

    results = []
    for i in range(k):
        word = tokenizer.decode([top_indices[i]])
        # Clean up the visualization (replace spaces with an underscore)
        #word = word.replace(" ", "_")
        conf = top_probs[i].item()
        results.append(f"'{word}' ({conf:.1%})")

    return " | ".join(results)

print(f"Prompt: '{prompt} [???]'\n")
print(f"{'LAYER':<10} | {'TOP 2 INTERNAL GUESSES'}")
print("-" * 100)

# We check Layer 0 (Input), Layer 6 (Middle), and Layer 12 (Output)
for i in [0, 2, 6, 10]:
    label = "Input" if i == 0 else f"Layer {i}"
    guesses = get_interpretable_guesses(i)
    print(f"{label:<10} | {guesses}")

Prompt: 'The capital of France is [???]'

LAYER      | TOP 5 INTERNAL GUESSES
----------------------------------------------------------------------------------------------------
Input      | ' destro' (53.3%) | ' mathemat' (28.0%)
Layer 2    | ' now' (27.1%) | ' not' (25.6%)
Layer 6    | ' now' (55.2%) | ' still' (10.9%)
Layer 10   | ' France' (62.1%) | ' Paris' (18.2%)


Observations

The "Blurry" Start: In the very first layer, the model's guess is often nonsensical (e.g., "the" or "a"). It hasn't "processed" the logic of the sentence yet; it's just looking at the most common words in English.

The "Brainstorming" Middle: By the middle layers (Layer 6), you might see the model start to guess words related to geography or countries. It has identified the context but hasn't reached the fact yet.

The "Clear" Conclusion: By the final layer (Layer 12), the model has refined the signals and usually outputs "Paris" with high confidence.


2023-2026 Research:

- Mechanistic Interpretability: This code shows that the "answer" is built incrementally. Researchers use this to find "knowledge neurons"—specific spots in specific layers where the fact "Paris = France" is actually stored.

- Hallucination Detection: If the confidence in Layer 12 is very low (e.g., only 10%), researchers know the model is likely about to hallucinate, even if it sounds confident in the text it eventually prints.

Exercise

- Do the same with other prompt in english "" and confirm the results.
  
- What happens when using a prompt in spanish ""?
